<h10>
This module asks you to build a small but complete GenAI service for Zepto: a document corpus you embed and index, a LangGraph-orchestrated flow that routes each query and retrieves grounded context, a structured-output guarantee, and a FastAPI wrapper you run locally. The entire pipeline is graded through a deterministic, fully offline mock mode for the LLM calls — no signup, no API key, and no network access to any LLM provider are required to earn full marks on this module. A real LLM call and a live cloud deployment are both optional, ungraded extensions layered on top of that graded baseline
</h10> <br>


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer

# --------------------------------------------------
# 1. Load the embedding model
# --------------------------------------------------

model = SentenceTransformer("all-MiniLM-L6-v2")

'''
Task-1
Load all 8 documents, chunk them (a simple per-document chunk, or a smaller fixed-size 
chunking scheme, is fine given their length), embed each chunk with all-MiniLM-L6-v2, and 
store the embeddings in a ChromaDB collection.
'''
with open("/home/leela_madhavi/Masai-AI_ML_Course-Batch2/CapstoneProject/support_assistant/docs/doc_01.txt", 'r') as delivery_policy:
    delivery_policy = delivery_policy.read()
with open("./docs/doc_02.txt", 'r') as return_and_refund:
    return_and_refund = return_and_refund.read()
with open("./docs/doc_03.txt", 'r') as membbership_tiers:
    membbership_tiers = membbership_tiers.read()
with open("./docs/doc_04.txt", 'r') as order_tracking:
    order_tracking = order_tracking.read()
with open("./docs/doc_05.txt", 'r') as order_cancel_policy:
    order_cancel_policy = order_cancel_policy.readlines()
with open("./docs/doc_06.txt", 'r') as damaged_or_missing_items:
    damaged_or_missing_items = damaged_or_missing_items.readlines()
with open("./docs/doc_07.txt", 'r') as gift_cards:
    gift_cards = gift_cards.readlines()
with open("./docs/doc_08.txt", 'r') as customer_support_hrs:
    customer_support_hrs = customer_support_hrs.readlines()


ModuleNotFoundError: No module named 'sentence_transformers'

In [13]:
!pip3 install groq chromadb -q

In [ ]:
from groq import Groq
import chromadb

def chunk_documents(text, source_name):
    #paragraph = doc_text.strip().split("\n\n")
    paragraphs = text.strip().split("\n\n")
    chunks = []
    for para in paragraphs:
        para = para.strip()
        chunks.append({
                'text': para,
                'source': source_name
            })

    return chunks

delivery_policy_chunks = chunk_documents(delivery_policy, "Delivery Policy")
return_and_refund_chunks = chunk_documents(delivery_policy, "Return and Refund")
membership_tiers_chunks = chunk_documents(delivery_policy, "Membership Tiers")
order_tracking_chunks = chunk_documents(delivery_policy, "Order Tracking")
gift_cards = chunk_documents(gift_cards, "Gift Cards")
order_cancel_policy = chunk_documents(order_cancel_policy, "Order Cancel Policy")
damaged_or_missing_items = chunk_documents(damaged_or_missing_items, "Damaged or Missing Items")
customer_support_hrs = chunk_documents(customer_support_hrs, "Customer Suppor Hours")

all_chunks = delivery_policy_chunks + return_and_refund_chunks + membership_tiers_chunks + order_tracking_chunks
print(f"All chunks size: {len(all_chunks)}")


All chunks size: 4


In [ ]:
# Storing the chunks in ChromaDb
#chroma_client = chromadb.Client()
# Create Chroma Client
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Create collection
collection = chroma_client.create_collection(name="company_docs")

# Prepare IDs and texts
ids = list(all_chunks.keys)
texts = list(all_chunks.values)

# Generate Embeddings
embeddings = model.encode(
    texts,
    convert_to_numpy=True
).tolist()

# Store documents + embeddings in ChromaDB
collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=[
        {"document_id": doc_id}
        for doc_id in ids
    ]
)

print(f"Stored {len(ids)} document chunks in ChromaDB.")

In [31]:
documents = []
ids = []
metadata = []

for i, chunk in enumerate(all_chunks):
  documents.append(chunk['text'])
  ids.append(f"chunk_{i}")
  metadata.append({"source": chunk["source"]})

collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadata
)

/home/leela_madhavi/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:20<00:00, 1.03MiB/s]  
